# JEE Student Dropout Prediction - Model Pipeline & Exploratory Analysis

## Overview

This notebook implements a comprehensive machine learning pipeline for predicting JEE (Joint Entrance Examination) student dropout risks. The model leverages a dual-dataset approach, merging academic performance metrics with behavioral and psychological indicators to identify at-risk students early.

### Feature Breakdown (17 Features)

**Academic Performance Metrics:**
- `attendance_rate`: Student's class attendance percentage
- `mock_test_avg`: Average score across mock examinations
- `physics_score`: Performance in Physics subject
- `chemistry_score`: Performance in Chemistry subject
- `maths_score`: Performance in Mathematics subject
- `mock_score_trend`: Trend in mock test performance over time
- `assignment_completion_rate`: Percentage of assignments completed
- `dpp_accuracy`: Daily Practice Problem accuracy rate
- `test_attempt_rate`: Rate of test participation

**Psychological & Behavioral Metrics:**
- `burnout_score`: Measured burnout level (1-10 scale)
- `stress_level`: Self-reported stress intensity (1-10 scale)
- `sleep_hours_avg`: Average daily sleep duration
- `study_hours_per_day`: Daily study time investment
- `study_consistency_score`: Consistency in study patterns
- `parental_pressure_level`: Perceived pressure from parents
- `peer_comparison_stress`: Stress from peer performance comparison
- `coaching_engagement_score`: Engagement level with coaching institute

### Target Objective

The binary target variable `dropout` indicates whether a student is at risk of discontinuing their JEE preparation (1 = Dropout Risk, 0 = No Dropout Risk). The model aims to achieve high recall for dropout prediction while maintaining strong overall accuracy to enable early intervention strategies.

In [ ]:
# Block 1: Introduction & Dependency Setup
# =========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import warnings
from pathlib import Path

# Scikit-learn dependencies
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import RocCurveDisplay

# Imbalanced learning
from imblearn.over_sampling import SMOTE

# XGBoost
import xgboost as xgb

# Configuration
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("✓ All dependencies imported successfully")
print("✓ Seaborn theme configured: whitegrid")

## Step 1: Ingesting Academic and Behavioral Datasets

We load the primary training dataset containing 8,000 student records with 17 features covering both academic performance and psychological metrics. The dataset includes a binary target variable indicating dropout risk status.

In [ ]:
# Block 2: Data Loading & Exploration
# ====================================

# Define paths
DATA_DIR = Path('../data')
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Load primary dataset (Dataset 2 - 8000 rows, 18 columns)
df = pd.read_csv(DATA_DIR / 'jee_training_data.csv')

# Display dataset information
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nFeature Columns ({len(df.columns) - 1}):")
for i, col in enumerate(df.columns[:-1], 1):
    print(f"  {i:2d}. {col}")

print(f"\nTarget Variable: {df.columns[-1]}")
print("\n" + "=" * 80)
print("FIRST 5 RECORDS")
print("=" * 80)
display(df.head())

# Check for missing values
print("\n" + "=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
    print("\nFilling missing values with median...")
    df = df.fillna(df.median())
else:
    print("✓ No missing values detected")

# Class distribution analysis
print("\n" + "=" * 80)
print("CLASS DISTRIBUTION")
print("=" * 80)
class_counts = df['dropout'].value_counts()
class_percentages = df['dropout'].value_counts(normalize=True) * 100

for label in [0, 1]:
    count = class_counts[label]
    pct = class_percentages[label]
    status = "No Dropout" if label == 0 else "Dropout Risk"
    print(f"  Class {label} ({status:15s}): {count:4d} samples ({pct:5.2f}%)")

# Visualize class distribution
fig, ax = plt.subplots(figsize=(10, 6))
sns.countplot(data=df, x='dropout', palette=['#6B5CE7', '#EF4444'], ax=ax)
ax.set_title('Class Distribution: Dropout Risk Status', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Dropout Status', fontsize=12, fontweight='semibold')
ax.set_ylabel('Count', fontsize=12, fontweight='semibold')
ax.set_xticklabels(['No Dropout', 'Dropout Risk'], fontsize=11)

# Add count labels on bars
for i, count in enumerate(class_counts):
    ax.text(i, count + 50, f'{count}\n({class_percentages[i]:.1f}%)', 
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# Feature statistics
print("\n" + "=" * 80)
print("FEATURE STATISTICS")
print("=" * 80)
display(df.describe())

## Step 2: Addressing Data Imbalance using Synthetic Minority Over-sampling Technique (SMOTE)

The dataset exhibits significant class imbalance with 70.5% non-dropout cases versus 29.5% dropout cases. To prevent model bias toward the majority class, we apply SMOTE (Synthetic Minority Over-sampling Technique) to generate synthetic samples for the minority class, achieving perfect balance at 5,638 samples per class.

In [ ]:
# Block 3: Class Balancing via SMOTE & Feature Scaling
# =====================================================

# Separate features and target
X = df.drop(columns=['dropout'])
y = df['dropout']

feature_names = list(X.columns)
print(f"Features: {len(feature_names)}")
print(f"Target distribution before SMOTE:")
print(f"  Class 0 (No Dropout): {sum(y == 0)}")
print(f"  Class 1 (Dropout):     {sum(y == 1)}")

# Apply SMOTE for class balancing
print("\nApplying SMOTE...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f"\nTarget distribution after SMOTE:")
print(f"  Class 0 (No Dropout): {sum(y_resampled == 0)}")
print(f"  Class 1 (Dropout):     {sum(y_resampled == 1)}")
print(f"\n✓ Classes perfectly balanced")

# Visualize balanced distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Before SMOTE
before_counts = [sum(y == 0), sum(y == 1)]
ax1.bar(['No Dropout', 'Dropout Risk'], before_counts, color=['#6B5CE7', '#EF4444'])
ax1.set_title('Before SMOTE', fontsize=12, fontweight='bold')
ax1.set_ylabel('Count', fontsize=11)
for i, count in enumerate(before_counts):
    ax1.text(i, count + 100, str(count), ha='center', va='bottom', fontweight='bold')

# After SMOTE
after_counts = [sum(y_resampled == 0), sum(y_resampled == 1)]
ax2.bar(['No Dropout', 'Dropout Risk'], after_counts, color=['#6B5CE7', '#EF4444'])
ax2.set_title('After SMOTE', fontsize=12, fontweight='bold')
ax2.set_ylabel('Count', fontsize=11)
for i, count in enumerate(after_counts):
    ax2.text(i, count + 100, str(count), ha='center', va='bottom', fontweight='bold')

plt.suptitle('Class Distribution Before and After SMOTE', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Train/Test split (80/20)
print("\n" + "=" * 80)
print("TRAIN/TEST SPLIT")
print("=" * 80)
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

print(f"Training set:   {X_train.shape[0]} samples ({len(X_train)/(len(X_train)+len(X_test))*100:.1f}%)")
print(f"Test set:       {X_test.shape[0]} samples ({len(X_test)/(len(X_train)+len(X_test))*100:.1f}%)")

# Feature scaling
print("\n" + "=" * 80)
print("FEATURE SCALING")
print("=" * 80)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ StandardScaler fitted on training data")
print("✓ Training data scaled")
print("✓ Test data scaled using fitted scaler")
print(f"\nScaled feature statistics (training set):")
print(f"  Mean: {X_train_scaled.mean():.6f}")
print(f"  Std:  {X_train_scaled.std():.6f}")

## Step 3: Training the XGBoost Classifier Engine

We employ XGBoost (eXtreme Gradient Boosting), a powerful gradient boosting framework known for its performance on tabular data. The model is configured with 200 estimators, maximum depth of 6, and optimized hyperparameters for dropout prediction. We utilize 5-fold stratified cross-validation to ensure robust performance estimation before final training.

In [ ]:
# Block 4: XGBoost Model Training & Cross-Validation
# ====================================================

# Initialize XGBoost classifier
print("=" * 80)
print("XGBOOST CLASSIFIER INITIALIZATION")
print("=" * 80)
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

print("Model Configuration:")
print(f"  n_estimators:      200")
print(f"  max_depth:         6")
print(f"  learning_rate:     0.1")
print(f"  subsample:         0.8")
print(f"  colsample_bytree:  0.8")
print(f"  random_state:      42")

# 5-fold stratified cross-validation
print("\n" + "=" * 80)
print("5-FOLD STRATIFIED CROSS-VALIDATION")
print("=" * 80)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')

print(f"\nFold-wise ROC-AUC Scores:")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")

print(f"\nCross-Validation Results:")
print(f"  Mean ROC-AUC:    {cv_scores.mean():.4f}")
print(f"  Std Dev:         {cv_scores.std():.4f}")
print(f"  95% CI:          {cv_scores.mean() - 1.96*cv_scores.std():.4f} to {cv_scores.mean() + 1.96*cv_scores.std():.4f}")

# Train on full training set
print("\n" + "=" * 80)
print("TRAINING ON FULL DATASET")
print("=" * 80)
xgb_model.fit(X_train_scaled, y_train)
print("✓ XGBoost model trained successfully")
print(f"✓ Training samples: {len(X_train)}")
print(f"✓ Features used: {len(feature_names)}")

## Step 4: Comprehensive Model Diagnostics & Interpretation

We evaluate the trained model on the held-out test set using multiple metrics. The confusion matrix provides insight into classification accuracy, while the ROC curve illustrates the trade-off between true positive and false positive rates across different thresholds.

In [ ]:
# Block 5: Interactive Visual Evaluation & Confusion Matrix
# ==========================================================

# Generate predictions on test set
print("=" * 80)
print("MODEL EVALUATION ON TEST SET")
print("=" * 80)
y_pred = xgb_model.predict(X_test_scaled)
y_pred_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\nTest Set Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  ROC-AUC:   {roc_auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {cm[0, 0]}")
print(f"  False Positives: {cm[0, 1]}")
print(f"  False Negatives: {cm[1, 0]}")
print(f"  True Positives:  {cm[1, 1]}")

# Visualization: Confusion Matrix Heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Dropout', 'Dropout Risk'],
            yticklabels=['No Dropout', 'Dropout Risk'],
            cbar_kws={'label': 'Count'},
            ax=ax1)
ax1.set_title('Confusion Matrix', fontsize=14, fontweight='bold', pad=20)
ax1.set_xlabel('Predicted Label', fontsize=12, fontweight='semibold')
ax1.set_ylabel('True Label', fontsize=12, fontweight='semibold')

# ROC Curve
RocCurveDisplay.from_estimator(xgb_model, X_test_scaled, y_test, ax=ax2)
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax2.set_title('ROC Curve', fontsize=14, fontweight='bold', pad=20)
ax2.set_xlabel('False Positive Rate', fontsize=12, fontweight='semibold')
ax2.set_ylabel('True Positive Rate', fontsize=12, fontweight='semibold')
ax2.legend(loc='lower right', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Model Diagnostics: Confusion Matrix & ROC Curve', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Classification Report
print("\n" + "=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)
print(classification_report(y_test, y_pred, target_names=['No Dropout', 'Dropout Risk'], digits=4))

## Step 5: Understanding What Influences Student Dropouts (Feature Importance)

Feature importance analysis reveals which variables contribute most to the model's predictions. This interpretability is crucial for understanding the key drivers of dropout risk and informing intervention strategies.

In [ ]:
# Block 6: Feature Importance Analysis
# =====================================

# Extract feature importances
importances = xgb_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("=" * 80)
print("FEATURE IMPORTANCE RANKING")
print("=" * 80)
print(f"\n{'Rank':<6} {'Feature':<35} {'Importance':<12} {'Percentage':<12}")
print("-" * 80)
for idx, row in feature_importance_df.iterrows():
    rank = feature_importance_df.index.get_loc(idx) + 1
    pct = (row['importance'] / importances.sum()) * 100
    print(f"{rank:<6} {row['feature']:<35} {row['importance']:<12.4f} {pct:<12.2f}%")

# Visualize feature importance
top_features = feature_importance_df.head(10)

fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(data=top_features, x='importance', y='feature', 
            palette='viridis', ax=ax)
ax.set_title('Top 10 Feature Importances for Dropout Prediction', 
          fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Feature Importance Score', fontsize=12, fontweight='semibold')
ax.set_ylabel('Feature Name', fontsize=12, fontweight='semibold')

# Add value labels
for i, v in enumerate(top_features['importance']):
    ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Feature importance by category
academic_features = ['attendance_rate', 'mock_test_avg', 'physics_score', 'chemistry_score', 
                      'maths_score', 'mock_score_trend', 'assignment_completion_rate', 
                      'dpp_accuracy', 'test_attempt_rate']
psychological_features = ['burnout_score', 'stress_level', 'sleep_hours_avg', 
                          'study_hours_per_day', 'study_consistency_score', 
                          'parental_pressure_level', 'peer_comparison_stress', 
                          'coaching_engagement_score']

academic_importance = feature_importance_df[feature_importance_df['feature'].isin(academic_features)]['importance'].sum()
psychological_importance = feature_importance_df[feature_importance_df['feature'].isin(psychological_features)]['importance'].sum()

print("\n" + "=" * 80)
print("IMPORTANCE BY FEATURE CATEGORY")
print("=" * 80)
print(f"\nAcademic Features:     {academic_importance:.4f} ({academic_importance/importances.sum()*100:.1f}%)")
print(f"Psychological Features: {psychological_importance:.4f} ({psychological_importance/importances.sum()*100:.1f}%)")

## Step 6: Serializing Operational Model State Pipeline

We export the trained model, scaler, feature list, and metadata to the artifacts directory for production deployment. These artifacts ensure reproducibility and enable the model to be loaded in the prediction API without retraining.

In [ ]:
# Block 7: Exporting Verified Artifacts
# =====================================

print("=" * 80)
print("SERIALIZING MODEL ARTIFACTS")
print("=" * 80)

# Save trained model
model_path = ARTIFACTS_DIR / 'model.pkl'
joblib.dump(xgb_model, model_path)
print(f"✓ Model saved to: {model_path}")

# Save scaler
scaler_path = ARTIFACTS_DIR / 'scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"✓ Scaler saved to: {scaler_path}")

# Save feature names
features_path = ARTIFACTS_DIR / 'features.json'
with open(features_path, 'w') as f:
    json.dump(feature_names, f, indent=2)
print(f"✓ Features saved to: {features_path}")

# Save metadata
metadata = {
    'model_type': 'XGBoost',
    'model_version': '1.0.0',
    'roc_auc': float(roc_auc),
    'features': feature_names,
    'n_features': len(feature_names),
    'n_samples_train': len(X_train),
    'n_samples_test': len(X_test),
    'cv_roc_auc_mean': float(cv_scores.mean()),
    'cv_roc_auc_std': float(cv_scores.std()),
    'test_metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'roc_auc': float(roc_auc)
    },
    'feature_importance': feature_importance_df.set_index('feature')['importance'].to_dict(),
    'training_date': pd.Timestamp.now().isoformat()
}

metadata_path = ARTIFACTS_DIR / 'metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Metadata saved to: {metadata_path}")

print("\n" + "=" * 80)
print("ARTIFACT EXPORT COMPLETE")
print("=" * 80)
print(f"\nArtifacts Directory: {ARTIFACTS_DIR}")
print("\nExported Files:")
print("  - model.pkl (Trained XGBoost model)")
print("  - scaler.pkl (StandardScaler for feature normalization)")
print("  - features.json (Feature name list for input validation)")
print("  - metadata.json (Model metadata and performance metrics)")

print("\n" + "=" * 80)
print("PIPELINE SUMMARY")
print("=" * 80)
print(f"\nModel: XGBoost Classifier")
print(f"Cross-Validation ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1 Score: {f1:.4f}")
print(f"\n✓ Model ready for production deployment")
print("=" * 80)